# 第005/136节 · 索引、广播与内存共享
CPU，依赖torch，种子42，小型合成数据，无需下载。
按顺序运行全部代码，再在本机重启并运行全部。
Notebook包含完整实现，不需要导入lesson.py。实际验证范围见validation.json。
运行前预测每个结果的形状和哪些对象会被修改。


## 1. 索引、切片与按元素计算
先写下形状与预期值，再运行核对。


In [ ]:
import torch

torch.manual_seed(42)
torch.set_num_threads(1)
device = torch.device("cpu")
X = torch.arange(12, dtype=torch.float32, device=device).reshape(3, 4)
row = X[1]
row_kept = X[1:2]
block = X[:2, 1:3]
print("PyTorch", torch.__version__, "device", device)
print("X", X.tolist())
print("row shapes", tuple(row.shape), tuple(row_kept.shape))
print("last value", X[-1, -1].item())
print("block", block.tolist())
print("doubled first row", (X * 2)[0].tolist())
assert row.shape == (4,) and row_kept.shape == (1, 4)
assert torch.equal(block, torch.tensor([[1., 2.], [5., 6.]], device=device))
assert (X * X).shape == X.shape


## 2. 逐轴判断广播
先写下形状与预期值，再运行核对。


In [ ]:
feature_offsets = torch.tensor([10., 20., 30., 40.], device=device)
shifted = X + feature_offsets
column = torch.tensor([[100.], [200.], [300.]], device=device)
row_offsets = torch.tensor([1., 2., 3., 4.], device=device)
grid = column + row_offsets
print("feature broadcast", tuple(shifted.shape), shifted.tolist())
print("outer broadcast", tuple(grid.shape), grid.tolist())
assert grid.shape == (3, 4)
assert grid[2, 3].item() == 304
try:
    X + torch.ones(3, device=device)
except RuntimeError:
    print("incompatible broadcast rejected: (3, 4) and (3,)")
else:
    raise AssertionError("Expected trailing axes 4 and 3 to conflict")


## 3. 别名、基本切片与clone
先写下形状与预期值，再运行核对。


In [ ]:
base = torch.arange(6, dtype=torch.float32, device=device).reshape(2, 3)
alias = base
view = base[:, 1:]
independent = view.clone()
view[0, 0] = 99
print("base after view edit", base.tolist())
print("clone after view edit", independent.tolist())
assert alias is base
assert base[0, 1].item() == 99
assert independent[0, 0].item() == 1


## 4. 新结果与原地修改
先写下形状与预期值，再运行核对。


In [ ]:
before = base.clone()
rebound = base + 10
assert torch.equal(base, before)
alias.add_(1)
print("base after in-place add", base.tolist())
print("view after in-place add", view.tolist())
print("new result kept", rebound.tolist())
assert base[0, 1].item() == 100
assert view[0, 0].item() == 100
assert independent[0, 0].item() == 1
assert rebound[0, 1].item() == 109
print("PASS")


## 独立练习
按homework.txt创建(4,3)矩阵并广播加偏移，修改基本切片并比较clone。
每次练习从新建数据开始，避免重复原地修改累积到旧状态。
先独立完成再看answers.html，保存个人练习副本。
